# 04 — Company Recommender / Router

> **Live product:** the hybrid + cosine blend (`0.7 * hybrid + 0.3 * cosine`) now runs in `glos_recommender.matching.match_companies` when embeddings exist. This notebook is an offline batch explorer that writes `recommendations.csv`.

**Maps from previous project:** Notebook 04 (intervention router + twin matching)

**Input:** clustered leavers + company feature store

**Tasks:**
1. Hybrid rule + similarity scoring (via `glos_recommender.matching`)
2. Cosine similarity on sentence-transformer vectors as an extra signal
3. Export top-3 recommendations per leaver

**Output:** `data/processed/recommendations.csv`, `app/app_data/recommendations.csv` (offline artefacts)  
**Production path:** Next.js → `POST /match` → `match_companies`

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from glos_recommender.matching import match_companies, match_reasons

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
APP_DATA_DIR = PROJECT_ROOT / "app" / "app_data"

leavers = pd.read_pickle(PROCESSED_DIR / "clustered_leavers.pkl")
companies = pd.read_pickle(PROCESSED_DIR / "company_feature_store.pkl")
print(f"Leavers: {len(leavers)} | Companies: {len(companies)}")

Leavers: 8 | Companies: 1139


## 1. Hybrid match (rules) + vector boost

In [2]:
company_matrix = np.stack(companies["vector"].values)

rows = []
for _, leaver_row in leavers.iterrows():
    form = leaver_row["form"]
    profile, ranked = match_companies(form, companies, top_n=len(companies))

    leaver_vec = np.array(leaver_row["vector"]).reshape(1, -1)
    sims = cosine_similarity(leaver_vec, company_matrix)[0]
    sim_map = dict(zip(companies["company_id"], sims))

    ranked = ranked.copy()
    ranked["cosine_sim"] = ranked["company_id"].map(sim_map)
    # Blend hybrid score with embedding similarity
    ranked["blended_score"] = 0.7 * ranked["final_score"] + 0.3 * ranked["cosine_sim"]
    top3 = ranked.sort_values("blended_score", ascending=False).head(3)

    for rank, (_, c) in enumerate(top3.iterrows(), start=1):
        reasons = match_reasons(profile, c)
        rows.append({
            "leaver_id": leaver_row["leaver_id"],
            "persona": leaver_row.get("persona", ""),
            "leaver_type": leaver_row["leaver_type"],
            "rank": rank,
            "company_id": c["company_id"],
            "company_name": c["name"],
            "town": c["town"],
            "blended_score": round(float(c["blended_score"]), 4),
            "hybrid_score": round(float(c["final_score"]), 4),
            "cosine_sim": round(float(c["cosine_sim"]), 4),
            "sector_score": c["sector_score"],
            "entry_score": c["entry_score"],
            "reasons": " | ".join(reasons),
            "website": c["website"],
        })

recommendations = pd.DataFrame(rows)
print(f"Recommendation rows: {len(recommendations)}")
recommendations.head(9)

Recommendation rows: 24


,leaver_id,persona,leaver_type,rank,company_id,company_name,town,blended_score,hybrid_score,cosine_sim,sector_score,entry_score,reasons,website
0,L001,Technical Specialist,School leaver (Year 11 / 13),1,VACC968FE65,AtkinsRéalis,Cheltenham,0.5088,0.4653,0.6102,0.7917,0.2500,Sector fit: cyber_digital | Entry routes they ...,
1,L001,Technical Specialist,School leaver (Year 11 / 13),2,VAC3CE1A642,CDS Defence & Security,Cheltenham,0.5040,0.4631,0.5994,0.7917,0.2500,Sector fit: cyber_digital | Entry routes they ...,
2,L001,Technical Specialist,School leaver (Year 11 / 13),3,VAC996B1AE3,St Roses School,Gloucestershire,0.5037,0.4757,0.5690,0.7917,0.3333,Sector fit: cyber_digital | Entry routes they ...,
3,L002,People & Care,School leaver (Year 11 / 13),1,VAC0B39C2E5,CANAL & RIVER TRUST,Gloucester,0.4982,0.4473,0.6169,0.8333,0.2500,"Sector fit: aerospace_manufacturing, construct...",
4,L002,People & Care,School leaver (Year 11 / 13),2,VACDC44CFCC,Scame UK,Gloucestershire,0.4950,0.4240,0.6608,0.4167,0.6667,Sector fit: aerospace_manufacturing | Entry ro...,
5,L002,People & Care,School leaver (Year 11 / 13),3,VACB1D120FE,HALFORDS AUTOCENTRES LIMITED,Gloucester,0.4924,0.4599,0.5681,0.4167,0.6667,Sector fit: aerospace_manufacturing | Entry ro...,
6,L003,Hands-on Maker,University undergraduate (final year / recent ...,1,GLC009,Farm491,Cirencester,0.3825,0.3210,0.5261,0.2917,0.5000,Sector fit: agri_tech_food | Entry routes they...,https://farm491.com
7,L003,Hands-on Maker,University undergraduate (final year / recent ...,2,VACD28C8196,MULLER UK & IRELAND GROUP LLP,Gloucestershire,0.3596,0.2816,0.5416,0.2917,0.2500,Sector fit: agri_tech_food | Entry routes they...,
8,L003,Hands-on Maker,University undergraduate (final year / recent ...,3,GLC010,Hartpury University and College,Hartpury,0.3489,0.3557,0.3329,0.2750,0.4000,"Sector fit: agri_tech_food, education_training...",https://www.hartpury.ac.uk


## 2. Inspect sample leaver

In [3]:
sample_id = recommendations["leaver_id"].iloc[0]
print(f"=== Top 3 for {sample_id} ===")
print(recommendations[recommendations["leaver_id"] == sample_id][
    ["rank", "company_name", "blended_score", "reasons"]
].to_string(index=False))

=== Top 3 for L001 ===
 rank           company_name  blended_score                                                                                                                                                                                                                                                                                   reasons
    1           AtkinsRéalis         0.5088 Sector fit: cyber_digital | Entry routes they offer that suit you: apprenticeship | Role families aligned with your work style: analyst, cyber, data, software | Work-style signals: Investigative, Conventional | Your interests: Cybersecurity & digital defence, Software / apps / web
    2 CDS Defence & Security         0.5040 Sector fit: cyber_digital | Entry routes they offer that suit you: apprenticeship | Role families aligned with your work style: analyst, cyber, data, software | Work-style signals: Investigative, Conventional | Your interests: Cybersecurity & digital defence, Software / apps / web

## 3. Save exports for Streamlit

In [4]:
out_csv = PROCESSED_DIR / "recommendations.csv"
recommendations.to_csv(out_csv, index=False)
recommendations.to_csv(APP_DATA_DIR / "recommendations.csv", index=False)

# Lightweight company table for the app (no vectors)
export_cols = [
    c for c in companies.columns
    if c not in {"vector", "sector_list", "entry_route_list", "role_family_list", "form"}
]
companies[export_cols].to_csv(APP_DATA_DIR / "companies_master.csv", index=False)

print(f"Saved {out_csv}")
print(f"Saved app/app_data/recommendations.csv")
print("\nProceed to Notebook 05 — RAG Briefing Generator.")

Saved c:\Users\MSI Katana Gaming\HigherED_ML_app\data\processed\recommendations.csv
Saved app/app_data/recommendations.csv

Proceed to Notebook 05 — RAG Briefing Generator.
